In [0]:
# ============================================================
# 07_HIERARCHY_CONSTRUCTION + GLOBAL COMPONENT RECONCILIATION
# Firmable | Rule-based | Ground truth is NOT used
#
# Inputs:
#   workspace.firmable_v2_silver.employees_clean
#   workspace.firmable_v2_silver.manager_assignments
#   workspace.firmable_v2_silver.manager_candidate_scores
#
# Outputs (existing downstream table names preserved):
#   workspace.firmable_v2_silver.manager_assignments
#   workspace.firmable_v2_silver.hierarchy_edges
#   workspace.firmable_v2_silver.employee_hierarchy
#
# Reconciliation policy:
# - Start from Notebook 06 assignments.
# - Identify disconnected components and their roots.
# - Attempt to attach an unassigned component root to a manager
#   already connected to that company's CEO component.
# - Only use scored, non-ambiguous candidates with score >= 55.
# - Require strict seniority, same-company references and remaining
#   manager capacity.
# - Reconciliation may use at most one additional direct-report slot
#   per manager; Notebook 06's original capacity policy is unchanged.
# - A cross-component link cannot create a cycle because its manager
#   is in a different component at the time of assignment.
# - Recompute components after each reconciliation pass.
# - Never force a link when no defensible candidate exists.
# - Ground truth is never read.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

EMPLOYEE_TABLE = "workspace.firmable_v2_silver.employees_clean"
ASSIGNMENT_TABLE = "workspace.firmable_v2_silver.manager_assignments"
CANDIDATE_TABLE = "workspace.firmable_v2_silver.manager_candidate_scores"

EDGE_TABLE = "workspace.firmable_v2_silver.hierarchy_edges"
HIERARCHY_TABLE = "workspace.firmable_v2_silver.employee_hierarchy"
ASSIGNMENT_BACKUP_TABLE = (
    "workspace.firmable_v2_silver.manager_assignments_pre_reconciliation"
)

MIN_SCORE = 55
MAX_HIERARCHY_DEPTH = 20
MAX_RECONCILIATION_PASSES = 10

# ============================================================
# 1. LOAD INPUTS AND VALIDATE REQUIRED COLUMNS
# ============================================================

employees = spark.table(EMPLOYEE_TABLE).select(
    "company_id",
    "employee_id",
    "company_name",
    "job_title_clean",
    "department",
    "business_unit",
    "team_id",
    "team_name",
    "inferred_seniority_level",
    "management_authority",
    "title_function"
).dropDuplicates(["company_id", "employee_id"])

assignments = spark.table(ASSIGNMENT_TABLE)
candidates = spark.table(CANDIDATE_TABLE)

required_assignment_columns = {
    "company_id", "employee_id", "candidate_manager_id",
    "candidate_score", "relationship_type", "assignment_status",
    "assignment_round"
}
required_candidate_columns = {
    "company_id", "employee_id", "candidate_manager_id",
    "candidate_score", "candidate_rank", "ambiguity_flag",
    "manager_seniority", "relationship_type"
}

missing_assignment_columns = required_assignment_columns - set(assignments.columns)
missing_candidate_columns = required_candidate_columns - set(candidates.columns)

if missing_assignment_columns:
    raise ValueError(
        "Missing manager_assignments columns: "
        f"{sorted(missing_assignment_columns)}"
    )
if missing_candidate_columns:
    raise ValueError(
        "Missing manager_candidate_scores columns: "
        f"{sorted(missing_candidate_columns)}"
    )

employee_count = employees.count()
assignment_employee_count = assignments.select(
    "company_id", "employee_id"
).dropDuplicates().count()

if employee_count != assignment_employee_count:
    raise ValueError(
        f"Employee/assignment key counts differ: "
        f"{employee_count} vs {assignment_employee_count}. "
        "Stop before hierarchy construction."
    )

duplicate_assignment_keys = (
    assignments.groupBy("company_id", "employee_id")
    .count().filter(F.col("count") > 1).count()
)
if duplicate_assignment_keys:
    raise ValueError(
        f"Found {duplicate_assignment_keys} duplicate employee assignment rows."
    )

# Preserve the exact Notebook 06 output before reconciliation.
if not spark.catalog.tableExists(ASSIGNMENT_BACKUP_TABLE):
    spark.sql(
        f"CREATE TABLE {ASSIGNMENT_BACKUP_TABLE} "
        f"DEEP CLONE {ASSIGNMENT_TABLE}"
    )
    print("Created assignment backup:", ASSIGNMENT_BACKUP_TABLE)
else:
    print("Existing assignment backup retained:", ASSIGNMENT_BACKUP_TABLE)

# Normalize assignment manager field for hierarchy processing.
current_assignments = assignments.withColumn(
    "manager_id", F.col("candidate_manager_id")
).persist(StorageLevel.MEMORY_AND_DISK)

# ============================================================
# 2. HELPER: BUILD COMPONENT MAP FROM CURRENT ASSIGNMENTS
# Every employee is mapped to the root of their current component.
# ============================================================

def build_component_map(assignment_df, employee_df):
    edge_df = (
        assignment_df
        .filter(F.col("manager_id").isNotNull())
        .select(
            "company_id",
            F.col("manager_id").alias("parent_id"),
            "employee_id"
        )
        .dropDuplicates()
    )

    root_frontier = (
        employee_df
        .select("company_id", "employee_id")
        .join(
            assignment_df.select(
                "company_id", "employee_id", "manager_id"
            ),
            ["company_id", "employee_id"],
            "left"
        )
        .filter(F.col("manager_id").isNull())
        .select(
            "company_id",
            "employee_id",
            F.col("employee_id").alias("root_employee_id"),
            F.lit(0).alias("hierarchy_depth")
        )
        .dropDuplicates()
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    visited_map = root_frontier
    frontier_map = root_frontier
    root_count_local = root_frontier.count()

    if root_count_local == 0:
        root_frontier.unpersist()
        raise ValueError(
            "No hierarchy roots found. A rootless component may contain a cycle."
        )

    hit_depth_limit = False

    for depth in range(1, MAX_HIERARCHY_DEPTH + 1):
        parent_frontier = frontier_map.select(
            "company_id",
            F.col("employee_id").alias("parent_id"),
            "root_employee_id",
            "hierarchy_depth"
        )

        next_frontier = (
            edge_df.alias("e")
            .join(
                parent_frontier.alias("p"),
                (F.col("e.company_id") == F.col("p.company_id")) &
                (F.col("e.parent_id") == F.col("p.parent_id")),
                "inner"
            )
            .select(
                F.col("e.company_id").alias("company_id"),
                F.col("e.employee_id").alias("employee_id"),
                F.col("p.root_employee_id").alias("root_employee_id"),
                (F.col("p.hierarchy_depth") + 1).alias("hierarchy_depth")
            )
            .join(
                visited_map.select("company_id", "employee_id"),
                ["company_id", "employee_id"],
                "left_anti"
            )
            .dropDuplicates(["company_id", "employee_id"])
            .persist(StorageLevel.MEMORY_AND_DISK)
        )

        new_count = next_frontier.count()

        if new_count == 0:
            next_frontier.unpersist()
            break

        old_visited = visited_map
        old_frontier = frontier_map

        visited_map = (
            visited_map.unionByName(next_frontier)
            .persist(StorageLevel.MEMORY_AND_DISK)
        )
        visited_map.count()

        old_visited.unpersist()
        if old_frontier is not old_visited:
            old_frontier.unpersist()

        frontier_map = next_frontier

        if depth == MAX_HIERARCHY_DEPTH:
            hit_depth_limit = True

    if frontier_map is not visited_map:
        frontier_map.unpersist()

    mapped_count = visited_map.select(
        "company_id", "employee_id"
    ).dropDuplicates().count()

    expected_count = employee_df.select(
        "company_id", "employee_id"
    ).dropDuplicates().count()

    if hit_depth_limit:
        visited_map.unpersist()
        raise ValueError(
            f"Hierarchy traversal reached depth {MAX_HIERARCHY_DEPTH}. "
            "Investigate before publishing."
        )

    if mapped_count != expected_count:
        visited_map.unpersist()
        raise ValueError(
            f"{expected_count - mapped_count} employees are unreachable "
            "from component roots. Possible cycle or invalid reference."
        )

    return visited_map

# ============================================================

# ============================================================
# 3. RECONCILE DISCONNECTED COMPONENTS
# Efficient progressive reconciliation using Silver rules only.
#
# IMPORTANT DESIGN:
# - Do not rebuild the full component map inside every pass.
# - A candidate manager must be strictly more senior than the root
#   employee (lower seniority number = more senior).
# - Existing assignments also obey strict seniority. Therefore a
#   same-component manager cannot be strictly more senior than a
#   component root; the seniority rule prevents a cycle.
# - Recompute root status and manager load each pass.
# - Permit one additional direct-report slot only in reconciliation.
# - Candidate rank fallback is naturally handled in later passes:
#   once a preferred manager is full, that manager is excluded and
#   the root can try its next eligible candidate.
# - Reconciliation updates the existing employee assignment row;
#   it never appends a second row for the same employee.
# ============================================================

reconciliation_log = []
total_reconciled = 0

for reconciliation_pass in range(1, MAX_RECONCILIATION_PASSES + 1):

    print(f"Starting reconciliation pass {reconciliation_pass}...")

    # --------------------------------------------------------
    # A. Current unassigned component roots
    # --------------------------------------------------------

    current_roots = (
        current_assignments
        .filter(F.col("manager_id").isNull())
        .select("company_id", "employee_id")
        .join(
            employees.select(
                "company_id",
                "employee_id",
                F.col("inferred_seniority_level")
                 .alias("employee_seniority_actual")
            ),
            ["company_id", "employee_id"],
            "inner"
        )
        .filter(
            F.col("employee_seniority_actual").isNotNull() &
            (F.col("employee_seniority_actual") != 0)
        )
    )

    root_count = current_roots.count()
    print(f"Unassigned roots before pass {reconciliation_pass}: {root_count}")

    if root_count == 0:
        print("No unassigned roots remain. Reconciliation completed.")
        break

    # --------------------------------------------------------
    # B. Current direct-report load and reconciliation capacity
    # --------------------------------------------------------

    current_manager_load = (
        current_assignments
        .filter(F.col("manager_id").isNotNull())
        .groupBy(
            "company_id",
            F.col("manager_id").alias("candidate_manager_id")
        )
        .agg(F.count("*").alias("current_direct_reports"))
    )

    manager_capacity = (
        employees.select(
            "company_id",
            F.col("employee_id").alias("candidate_manager_id"),
            F.col("inferred_seniority_level")
             .alias("manager_seniority_actual")
        )
        .withColumn(
            "base_manager_capacity",
            F.when(F.col("manager_seniority_actual") <= 0, 8)
             .when(F.col("manager_seniority_actual") <= 7, 10)
             .when(F.col("manager_seniority_actual") <= 9, 8)
             .otherwise(6)
        )
        .withColumn(
            "manager_capacity",
            F.col("base_manager_capacity") + F.lit(1)
        )
        .join(
            current_manager_load,
            ["company_id", "candidate_manager_id"],
            "left"
        )
        .withColumn(
            "current_direct_reports",
            F.coalesce(
                F.col("current_direct_reports"),
                F.lit(0)
            )
        )
        .withColumn(
            "remaining_capacity",
            F.col("manager_capacity") -
            F.col("current_direct_reports")
        )
        .filter(F.col("remaining_capacity") > 0)
    )

    # --------------------------------------------------------
    # C. Eligible candidate links for current roots
    # --------------------------------------------------------

    manager_seniority = employees.select(
        "company_id",
        F.col("employee_id").alias("candidate_manager_id"),
        F.col("inferred_seniority_level").alias("manager_seniority_actual")
    )

    root_candidates = (
        candidates
        .filter(
            (F.col("candidate_score") >= MIN_SCORE) &
            (F.col("ambiguity_flag") == F.lit(False))
        )
        .join(
            current_roots,
            ["company_id", "employee_id"],
            "inner"
        )
        .join(
            manager_seniority,
            ["company_id", "candidate_manager_id"],
            "inner"
        )
        .filter(
            F.col("candidate_manager_id") != F.col("employee_id")
        )
        # Lower seniority number means more senior.
        .filter(
            F.col("manager_seniority_actual") <
            F.col("employee_seniority_actual")
        )
        .join(
            manager_capacity.select(
                "company_id",
                "candidate_manager_id",
                "manager_capacity",
                "current_direct_reports",
                "remaining_capacity"
            ),
            ["company_id", "candidate_manager_id"],
            "inner"
        )
        .dropDuplicates([
            "company_id",
            "employee_id",
            "candidate_manager_id"
        ])
    )

    eligible_count = root_candidates.count()
    print(
        f"Eligible root-manager options in pass "
        f"{reconciliation_pass}: {eligible_count}"
    )

    if eligible_count == 0:
        reconciliation_log.append((reconciliation_pass, 0))
        print("No eligible candidate links remain. Stopping reconciliation.")
        break

    # --------------------------------------------------------
    # D. Each root proposes its best currently available manager
    # --------------------------------------------------------

    employee_choice_window = (
        Window
        .partitionBy("company_id", "employee_id")
        .orderBy(
            F.col("candidate_rank").asc_nulls_last(),
            F.col("candidate_score").desc(),
            F.col("candidate_manager_id").asc()
        )
    )

    best_root_options = (
        root_candidates
        .withColumn(
            "employee_option_rank",
            F.row_number().over(employee_choice_window)
        )
        .filter(F.col("employee_option_rank") == 1)
    )

    # --------------------------------------------------------
    # E. Resolve competition for each manager's remaining slots
    # --------------------------------------------------------

    manager_competition_window = (
        Window
        .partitionBy("company_id", "candidate_manager_id")
        .orderBy(
            F.col("candidate_score").desc(),
            F.col("candidate_rank").asc_nulls_last(),
            F.col("employee_id").asc()
        )
    )

    accepted_links = (
        best_root_options
        .withColumn(
            "manager_slot_rank",
            F.row_number().over(manager_competition_window)
        )
        .filter(
            F.col("manager_slot_rank") <=
            F.col("remaining_capacity")
        )
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    accepted_count = accepted_links.count()

    print(
        f"Reconciliation pass {reconciliation_pass}: "
        f"{accepted_count} accepted links"
    )

    reconciliation_log.append(
        (reconciliation_pass, accepted_count)
    )

    if accepted_count == 0:
        accepted_links.unpersist()
        break

    # --------------------------------------------------------
    # F. Construct new assignment rows using existing schema
    # --------------------------------------------------------

    accepted_links = (
        accepted_links
        .withColumn(
            "assignment_round",
            F.lit(100 + reconciliation_pass)
        )
        .withColumn(
            "assignment_status",
            F.lit("ASSIGNED")
        )
        .withColumn(
            "assignment_method",
            F.lit("COMPONENT_RECONCILIATION")
        )
        .withColumn(
            "assignment_reason",
            F.lit("DEFENSIBLE_SENIORITY_AND_CAPACITY_LINK")
        )
        .withColumn(
            "candidate_count",
            F.lit(None).cast("int")
        )
        .withColumn(
            "best_candidate_score",
            F.col("candidate_score")
        )
        .withColumn(
            "is_ambiguous",
            F.lit(0)
        )
        .withColumn(
            "maximum_candidate_rank",
            F.lit(None).cast("int")
        )
        .withColumn(
            "candidates_meeting_minimum_score",
            F.lit(None).cast("int")
        )
    )

    new_assignment_expressions = []

    for field in current_assignments.schema.fields:

        name = field.name

        if name == "manager_id":
            continue

        if name in accepted_links.columns:
            new_assignment_expressions.append(
                F.col(name).cast(field.dataType).alias(name)
            )
        elif name == "candidate_manager_id":
            new_assignment_expressions.append(
                F.col("candidate_manager_id")
                 .cast(field.dataType)
                 .alias(name)
            )
        else:
            new_assignment_expressions.append(
                F.lit(None).cast(field.dataType).alias(name)
            )

    new_assignment_rows = accepted_links.select(
        *new_assignment_expressions
    ).withColumn("_reconciliation_match", F.lit(1))

    # IMPORTANT:
    # manager_assignments already contains exactly one row per employee,
    # including employees whose manager is currently NULL. Reconciliation
    # must UPDATE those existing rows, not UNION additional employee rows.
    # A UNION would create duplicate employee keys and inflate the hierarchy
    # row count by exactly the number of accepted reconciliation links.
    current_assignment_columns = [
        c for c in current_assignments.columns
        if c != "manager_id"
    ]

    current_alias = current_assignments.select(
        *current_assignment_columns
    ).alias("current")
    updates_alias = new_assignment_rows.alias("updates")

    joined_assignments = current_alias.join(
        updates_alias,
        ["company_id", "employee_id"],
        "left"
    )

    updated_assignment_expressions = []

    for field in current_assignments.schema.fields:
        name = field.name

        if name == "manager_id":
            continue

        updated_assignment_expressions.append(
            F.when(
                F.col("updates._reconciliation_match") == 1,
                F.col(f"updates.{name}")
            )
            .otherwise(F.col(f"current.{name}"))
            .cast(field.dataType)
            .alias(name)
        )

    current_assignments_next = (
        joined_assignments
        .select(*updated_assignment_expressions)
        .withColumn(
            "manager_id",
            F.col("candidate_manager_id")
        )
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    # Materialize the updated one-row-per-employee snapshot before
    # releasing cached inputs.
    next_assignment_count = current_assignments_next.count()

    if next_assignment_count != employee_count:
        current_assignments_next.unpersist()
        accepted_links.unpersist()
        raise ValueError(
            "Reconciliation changed the assignment row count: "
            f"{next_assignment_count} vs expected {employee_count}. "
            "Stopping before publication."
        )

    accepted_links.unpersist()
    current_assignments.unpersist()
    current_assignments = current_assignments_next

    total_reconciled += accepted_count

    print(
        f"Pass {reconciliation_pass} complete. "
        f"Assignment rows now: {next_assignment_count}; "
        f"total reconciliation links: {total_reconciled}"
    )

print("\\n========== RECONCILIATION SUMMARY ==========")
print("Accepted reconciliation links:", total_reconciled)
print("Pass-wise results:", reconciliation_log)

# ============================================================
# 4. FINAL COMPONENT MAP AND HIERARCHY BUILD
# ============================================================

final_component_map = build_component_map(current_assignments, employees)

base = (
    employees
    .join(
        current_assignments.select(
            "company_id",
            "employee_id",
            "candidate_manager_id",
            "candidate_score",
            "relationship_type",
            "assignment_status",
            "assignment_round",
            *(
                ["assignment_method"] if "assignment_method" in current_assignments.columns else []
            ),
            *(
                ["assignment_reason"] if "assignment_reason" in current_assignments.columns else []
            )
        ),
        ["company_id", "employee_id"],
        "left"
    )
    .withColumn("manager_id", F.col("candidate_manager_id"))
    .join(
        final_component_map.select(
            "company_id", "employee_id",
            "root_employee_id", "hierarchy_depth"
        ),
        ["company_id", "employee_id"],
        "left"
    )
    .withColumn("is_root", F.col("manager_id").isNull())
    .withColumn(
        "root_type",
        F.when(
            F.col("is_root") & (F.col("inferred_seniority_level") == 0),
            F.lit("CEO_ROOT")
        )
        .when(F.col("is_root"), F.lit("UNASSIGNED_ROOT"))
        .otherwise(F.lit("CHILD"))
    )
    .withColumn(
        "assignment_method",
        F.coalesce(F.col("assignment_method"), F.lit("ROOT_OR_UNASSIGNED"))
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# ============================================================
# 5. FINAL VALIDATION — NO DATA LOSS, DUPLICATES, INVALID LINKS
# ============================================================

final_total = base.count()
if final_total != employee_count:
    raise ValueError(
        f"Hierarchy row count {final_total} differs from employee count {employee_count}."
    )

duplicate_hierarchy_rows = (
    base.groupBy("company_id", "employee_id")
    .count().filter(F.col("count") > 1).count()
)

duplicate_manager_assignments = (
    base.filter(F.col("manager_id").isNotNull())
    .groupBy("company_id", "employee_id")
    .count().filter(F.col("count") > 1).count()
)

employee_keys = employees.select(
    "company_id", F.col("employee_id").alias("checked_employee_id"),
    F.col("inferred_seniority_level").alias("employee_level")
)
manager_keys = employees.select(
    "company_id", F.col("employee_id").alias("checked_manager_id"),
    F.col("inferred_seniority_level").alias("manager_level")
)

manager_reference_check = (
    base.filter(F.col("manager_id").isNotNull()).alias("h")
    .join(
        manager_keys.alias("m"),
        (F.col("h.company_id") == F.col("m.company_id")) &
        (F.col("h.manager_id") == F.col("m.checked_manager_id")),
        "left"
    )
)

invalid_manager_references = manager_reference_check.filter(
    F.col("m.checked_manager_id").isNull()
).count()

seniority_check = (
    base.filter(F.col("manager_id").isNotNull()).alias("h")
    .join(
        employee_keys.alias("e"),
        (F.col("h.company_id") == F.col("e.company_id")) &
        (F.col("h.employee_id") == F.col("e.checked_employee_id")),
        "inner"
    )
    .join(
        manager_keys.alias("m"),
        (F.col("h.company_id") == F.col("m.company_id")) &
        (F.col("h.manager_id") == F.col("m.checked_manager_id")),
        "inner"
    )
)

invalid_seniority = seniority_check.filter(
    F.col("m.manager_level") >= F.col("e.employee_level")
).count()

# Validate capacity after reconciliation.
manager_capacity_check = (
    employees.select(
        "company_id",
        F.col("employee_id").alias("candidate_manager_id"),
        F.col("inferred_seniority_level").alias("manager_seniority_actual")
    )
    .withColumn(
        "base_allowed_capacity",
        F.when(F.col("manager_seniority_actual") <= 0, 8)
         .when(F.col("manager_seniority_actual") <= 7, 10)
         .when(F.col("manager_seniority_actual") <= 9, 8)
         .otherwise(6)
    )
    # Final validation mirrors the one-slot reconciliation extension.
    .withColumn(
        "allowed_capacity",
        F.col("base_allowed_capacity") + F.lit(1)
    )
)

capacity_violations = (
    base.filter(F.col("manager_id").isNotNull())
    .groupBy("company_id", F.col("manager_id").alias("candidate_manager_id"))
    .agg(F.count("*").alias("actual_direct_reports"))
    .join(
        manager_capacity_check,
        ["company_id", "candidate_manager_id"],
        "inner"
    )
    .filter(F.col("actual_direct_reports") > F.col("allowed_capacity"))
    .count()
)

unreachable_count = base.filter(
    F.col("root_employee_id").isNull() |
    F.col("hierarchy_depth").isNull()
).count()

self_relationships = base.filter(
    F.col("manager_id") == F.col("employee_id")
).count()

print("\n========== GLOBAL RECONCILIATION VALIDATION ==========")
print("Employees:", employee_count)
print("Accepted component reconciliation links:", total_reconciled)
print("Duplicate hierarchy rows:", duplicate_hierarchy_rows)
print("Duplicate manager assignments:", duplicate_manager_assignments)
print("Invalid manager references:", invalid_manager_references)
print("Invalid seniority relationships:", invalid_seniority)
print("Capacity violations:", capacity_violations)
print("Self relationships:", self_relationships)
print("Unreachable employees:", unreachable_count)

if any([
    final_total != employee_count,
    duplicate_hierarchy_rows > 0,
    duplicate_manager_assignments > 0,
    invalid_manager_references > 0,
    invalid_seniority > 0,
    capacity_violations > 0,
    self_relationships > 0,
    unreachable_count > 0
]):
    raise ValueError(
        "Final hierarchy validation failed. Existing published tables were NOT overwritten."
    )

# ============================================================
# 6. BUILD EDGE TABLE
# ============================================================

edges = base.filter(
    F.col("manager_id").isNotNull()
).select(
    "company_id",
    "manager_id",
    "employee_id",
    "candidate_score",
    "relationship_type",
    "assignment_round",
    "assignment_status",
    "assignment_method"
)

# ============================================================
# 7. BACKUP CURRENT PUBLISHED TABLES BEFORE REPLACEMENT
# ============================================================

for source_table, backup_table in [
    (
        HIERARCHY_TABLE,
        "workspace.firmable_v2_silver.employee_hierarchy_pre_reconciliation"
    ),
    (
        EDGE_TABLE,
        "workspace.firmable_v2_silver.hierarchy_edges_pre_reconciliation"
    )
]:
    if spark.catalog.tableExists(source_table) and not spark.catalog.tableExists(backup_table):
        spark.sql(f"CREATE TABLE {backup_table} DEEP CLONE {source_table}")
        print("Created backup:", backup_table)

# ============================================================
# 8. PUBLISH RECONCILED ASSIGNMENTS, EDGES AND HIERARCHY
# All validations have passed before any overwrite.
# ============================================================

assignment_to_publish = current_assignments.select(
    *[c for c in current_assignments.columns if c != "manager_id"]
)

(
    assignment_to_publish.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ASSIGNMENT_TABLE)
)

(
    edges.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(EDGE_TABLE)
)

(
    base.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(HIERARCHY_TABLE)
)

# ============================================================
# 9. FINAL SUMMARY
# ============================================================

final_hierarchy = spark.table(HIERARCHY_TABLE)

assigned_count = final_hierarchy.filter(
    F.col("manager_id").isNotNull()
).count()

component_roots = final_hierarchy.filter(
    F.col("is_root")
).count()

ceo_roots = final_hierarchy.filter(
    F.col("root_type") == "CEO_ROOT"
).count()

unassigned_roots = final_hierarchy.filter(
    F.col("root_type") == "UNASSIGNED_ROOT"
).count()

maximum_depth = final_hierarchy.agg(
    F.max("hierarchy_depth").alias("max_depth")
).first()["max_depth"]

print("\n========== HIERARCHY CONSTRUCTION COMPLETED ==========")
print("Total employees:", final_total)
print("Assigned employees:", assigned_count)
print("Unassigned employees:", final_total - assigned_count)
print("Component roots:", component_roots)
print("CEO roots:", ceo_roots)
print("Unassigned roots:", unassigned_roots)
print("Accepted reconciliation links:", total_reconciled)
print("Maximum hierarchy depth:", maximum_depth)
print("Unreachable employees:", unreachable_count)

print("\nROOT TYPE DISTRIBUTION")
final_hierarchy.filter(
    F.col("is_root")
).groupBy("root_type").count().orderBy("root_type").show(truncate=False)

print("\nHIERARCHY DEPTH DISTRIBUTION")
final_hierarchy.groupBy(
    "hierarchy_depth"
).count().orderBy("hierarchy_depth").show(truncate=False)

print("\nASSIGNMENT METHOD DISTRIBUTION")
final_hierarchy.groupBy(
    "assignment_method"
).count().orderBy("assignment_method").show(truncate=False)

print("\nOUTPUT TABLES:")
print(ASSIGNMENT_TABLE)
print(EDGE_TABLE)
print(HIERARCHY_TABLE)
print("Ground truth was not read or used.")
print("ALL HIERARCHY VALIDATIONS PASSED")

# ============================================================
# 10. RELEASE CACHED DATAFRAMES
# ============================================================

current_assignments.unpersist()
final_component_map.unpersist()
base.unpersist()


In [0]:
%sql
WITH unassigned_roots AS (
    SELECT
        company_id,
        employee_id,
        root_employee_id AS employee_component_root
    FROM workspace.firmable_v2_silver.employee_hierarchy
    WHERE is_root = true
      AND root_type = 'UNASSIGNED_ROOT'
),
valid_candidates AS (
    SELECT
        c.company_id,
        c.employee_id,
        c.candidate_manager_id,
        me.inferred_seniority_level AS manager_seniority,
        mh.root_employee_id AS manager_component_root,
        r.employee_component_root,
        CASE
            WHEN me.inferred_seniority_level <= 0 THEN 8
            WHEN me.inferred_seniority_level <= 7 THEN 10
            WHEN me.inferred_seniority_level <= 9 THEN 8
            ELSE 6
        END AS manager_capacity
    FROM unassigned_roots r
    JOIN workspace.firmable_v2_silver.manager_candidate_scores c
      ON r.company_id = c.company_id
     AND r.employee_id = c.employee_id
    JOIN workspace.firmable_v2_silver.employees_clean ee
      ON c.company_id = ee.company_id
     AND c.employee_id = ee.employee_id
    JOIN workspace.firmable_v2_silver.employees_clean me
      ON c.company_id = me.company_id
     AND c.candidate_manager_id = me.employee_id
    JOIN workspace.firmable_v2_silver.employee_hierarchy mh
      ON c.company_id = mh.company_id
     AND c.candidate_manager_id = mh.employee_id
    WHERE c.candidate_score >= 55
      AND c.ambiguity_flag = false
      AND me.inferred_seniority_level < ee.inferred_seniority_level
      AND mh.root_employee_id <> r.employee_component_root
),
manager_load AS (
    SELECT
        company_id,
        candidate_manager_id,
        COUNT(*) AS current_reports
    FROM workspace.firmable_v2_silver.manager_assignments
    WHERE assignment_status = 'ASSIGNED'
    GROUP BY company_id, candidate_manager_id
),
manager_capacity_summary AS (
    SELECT
        v.company_id,
        v.candidate_manager_id,
        v.manager_capacity,
        COALESCE(l.current_reports, 0) AS current_reports,
        COUNT(DISTINCT v.employee_id) AS candidate_roots
    FROM valid_candidates v
    LEFT JOIN manager_load l
      ON v.company_id = l.company_id
     AND v.candidate_manager_id = l.candidate_manager_id
    GROUP BY
        v.company_id,
        v.candidate_manager_id,
        v.manager_capacity,
        COALESCE(l.current_reports, 0)
)
SELECT
    manager_capacity,
    current_reports,
    manager_capacity - current_reports AS remaining_capacity,
    COUNT(*) AS manager_count,
    SUM(candidate_roots) AS candidate_root_options
FROM manager_capacity_summary
GROUP BY manager_capacity, current_reports
ORDER BY remaining_capacity DESC, manager_capacity;